In [1]:
# 导包
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import URL
import os
from dotenv import load_dotenv

In [6]:
# 加载配置到当前进程并创建连接引擎
load_dotenv(r"D:\my_project\ecommerce-business-analytics\.env")
engine = create_engine(URL.create(
    drivername="mysql+pymysql",
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 3306)),
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    query={"charset": "utf8mb4"}
))

In [7]:
# 1.准备数据
data = pd.read_sql('select * from v_sales_transactions where invoice_date >= "2010-12-01" and invoice_date < "2011-12-01" and stock_code in ("20725", "22383", "20726")', engine)

In [26]:
# 2.计算各个商品对应的总销量
total_sales = data.groupby('stock_code',as_index=False)['quantity'].sum().rename(columns={'quantity':'total_sales'})
total_sales

,stock_code,total_sales
0,20725,19156
1,20726,9737
2,22383,12391


In [30]:
# 3.计算有效客户编号对应的销量
valid_customer = data.query('customer_id.notna()').groupby('stock_code',as_index= False)['quantity'].sum().rename(columns={'quantity':'valid_quantity'})
valid_customer

,stock_code,valid_quantity
0,20725,17540
1,20726,8945
2,22383,11604


In [20]:
# 4.计算客户编号为空对应的销量
invalid_customer = data.query('customer_id.isna()').groupby('stock_code')['quantity'].sum().reset_index(name = 'invalid_quantity')
invalid_customer

,stock_code,invalid_quantity
0,20725,1616
1,20726,792
2,22383,787


In [32]:
# 5.合并表
t1 = total_sales.merge(valid_customer, on='stock_code',how='inner')
t2 = t1.merge(invalid_customer, on='stock_code',how='inner')

In [38]:
# 6.计算有客户编号销量占比
t2['valid_ratio'] = ((t2['valid_quantity'] / t2['total_sales']) * 100).round(2)
finall_table = (t2.merge(data,on='stock_code',how='inner')).drop_duplicates(subset='stock_code',keep='first')[['description','stock_code','total_sales','valid_quantity','invalid_quantity','valid_ratio']]
finall_table

,description,stock_code,total_sales,valid_quantity,invalid_quantity,valid_ratio
0,LUNCH BAG RED RETROSPOT,20725,19156,17540,1616,91.56
1551,LUNCH BAG WOODLAND,20726,9737,8945,792,91.87
2558,LUNCH BAG SUKI DESIGN,22383,12391,11604,787,93.65


In [57]:
# 7.计算出该商品的有效客户id的前5大客户购买总量
red_sales = data.query('stock_code == "20725" & customer_id.notna()').sort_values('quantity', ascending = False).groupby('customer_id', as_index=False)['quantity'].sum().sort_values('quantity', ascending = False).head(5)['quantity'].sum()
red_sales

np.int64(3429)

In [61]:
# 8.计算已确认的贡献
red = finall_table.query('stock_code == "20725"')
red['confirmed'] = ((red_sales / red['total_sales']) * 100).round(2)
red

,description,stock_code,total_sales,valid_quantity,invalid_quantity,valid_ratio,confirmed
0,LUNCH BAG RED RETROSPOT,20725,19156,17540,1616,91.56,17.9


In [64]:
# 9.假设情景（假设缺失的客户id的订单全部来自这前五的大客户）
hypothesis = finall_table.query('stock_code == "20725"')
hypothesis['postulate'] = (((red_sales + hypothesis['invalid_quantity']) / hypothesis['total_sales']) * 100).round(2)
hypothesis

,description,stock_code,total_sales,valid_quantity,invalid_quantity,valid_ratio,postulate
0,LUNCH BAG RED RETROSPOT,20725,19156,17540,1616,91.56,26.34


In [69]:
# 10 合并表并对比
comparison = red.merge(hypothesis, on='stock_code',how='inner')[['description_x','stock_code','total_sales_x','confirmed','postulate']]
comparison

,description_x,stock_code,total_sales_x,confirmed,postulate
0,LUNCH BAG RED RETROSPOT,20725,19156,17.9,26.34


### 给销售经理的建议

建议优先对红色波点午餐袋（LUNCH BAG RED RETROSPOT）做小规模复购推广测试。观察期内，三款候选商品中，这款的可识别购买客户最多（524 名），其中 272 名至少购买过两次，适合先向只购买过一次的客户测试再次购买提醒。

在这款商品的 19,156 件总销量中，交易记录能确认购买量最大的五位可识别客户合计购买了 3,429 件，占 **17.90%**。另有 1,616 件缺少客户编号。如果全部由这五位客户购买，他们的占比会升至 **26.34%**。这部分销量的归属仍待确认，因此建议保留小规模试点安排，待比较收到提醒与未收到提醒的客户在同期的再次购买比例后，再决定是否扩大推广。